# W14 — Ablation and Sensitivity Studies

**Purpose.** Check that the design choices of the framework are justified and that the main conclusions of W12 do not depend on arbitrary settings. Each study changes **one** setting and keeps everything else as in W12 (same fleet FD001 + FD003, folds, truncation points, inner validation, W13 hyperparameters, collapse guard).

| Study | Settings compared | Feature sets | Question |
|---|---|---|---|
| A. Attention | dual attention (reference) / feature attention only / temporal attention only / no attention (mean pooling) | S, S+T | Does each attention mechanism improve the BiLSTM and the hybrid? |
| B. Feature selection | none / correlation-based / AFICv global / AFICv stratified, applied to both branches; reference = W12 design (correlation for ML, stratified AFICv for DL) | S+T+Op+A | Does BI-aware selection matter when BI is present? |
| C. Window length W | 15 / 30 (reference) / 45 / 60 cycles | S, S+T | Is W = 30 a reasonable choice? Does the gain of operating age depend on W? |
| D. RUL cap | 100 / 125 (reference) / 150 cycles | S, S+T | How much of the gain of operating age comes from the cap (early life is easy to predict from age when the RUL is capped)? |

**Evaluation set.** Truncation points are redrawn with at least 60 observed cycles (the largest window), so no test window is zero-padded and all studies share the same points. W14 numbers are therefore not directly comparable with W12.

**Cost control.** Seed 42 only (W12 showed SDs over seeds of 0.1–0.7 cycle for the hybrid), XGBoost + BiLSTM + α fusion (no Random Forest, dense fusion, SHAP or permutation importance, which W12 already covers). Hyperparameters are those selected by W13 for each feature set and are **not** re-tuned per variant (the reference is therefore favoured slightly; stated as a limitation). Reference runs are recomputed here with the same light pipeline so that every comparison is like for like.

**Statistics.** Each variant is compared with the reference of the same feature set on engine-level errors: ΔRMSE with bootstrap 95% CI (2,000 resamples of engines) and a **two-sided** Wilcoxon signed-rank test (an ablation can go either way), Holm-corrected within each study. For study D the key quantity is the gain of operating age (S+T − S) at each cap, with its CI.

Every (variant, feature set, fold) is cached on Drive; an interrupted run resumes. Use a GPU runtime.

## 0. Setup and data (same code as W12)

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Fresh clone every run; cache and results live on Drive so finished work survives a disconnect
    if os.path.exists('/content/PhD-Project-'):
        !rm -rf /content/PhD-Project-
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
    !pip install xgboost shap -q
    os.chdir('/content/PhD-Project-/src')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/PhD-Project-')
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
    SAVE_DIR = Path(f'/content/drive/MyDrive/PhD_results/W14_{TIMESTAMP}')
    CACHE_ROOT = Path('/content/drive/MyDrive/PhD_results/W14_cache')
else:
    PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    SAVE_DIR = None
    CACHE_ROOT = PROJECT_ROOT / 'results' / 'W14_cache'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

CMAPSS_DIR = PROJECT_ROOT / 'data' / 'raw_data' / 'C_MAPSS'
BI_DIR     = PROJECT_ROOT / 'data' / 'raw_data' / 'BI_data'
ASSET_DIR  = BI_DIR / 'asset'
# ---- Fleet: FD001 alone, or FD001 + FD003 analysed as ONE fleet of 200 engines ----
# FD003: same single operating condition as FD001, two fault modes (HPC + fan).
# Unit ids of FD003 are offset by +1000. Set DATASETS = ['FD001'] to reproduce the FD001-only runs.
DATASETS = ['FD001', 'FD003']
for _ds in DATASETS:
    assert (CMAPSS_DIR / f'train_{_ds}.txt').exists(), f"train_{_ds}.txt missing in {CMAPSS_DIR}: upload it to the repo"
FLEET_TAG = '_'.join(DATASETS)
TAG_SUFFIX = '' if FLEET_TAG == 'FD001' else f'_{FLEET_TAG}'   # FD001 keeps its original paths and caches

RESULTS    = PROJECT_ROOT / 'results' / f'W14{TAG_SUFFIX}'
CACHE_ROOT = CACHE_ROOT.parent / f'{CACHE_ROOT.name}{TAG_SUFFIX}'
if SAVE_DIR is not None:
    SAVE_DIR = SAVE_DIR.parent / f'{SAVE_DIR.name}{TAG_SUFFIX}'
for d in (RESULTS, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)
if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Fleet: {FLEET_TAG}\nProject root: {PROJECT_ROOT}\nResults: {RESULTS}\nCache: {CACHE_ROOT}\nDrive save: {SAVE_DIR}")

In [ ]:
# Regenerate both BI blocks from the generators in the repo, so data and code always match
for script in ['generate_bi_data.py', 'generate_bi_asset_data.py']:
    res = subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / script)],
                         capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(f"--- {script}: {'OK' if res.returncode == 0 else 'FAILED'}")
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError(f"{script} failed")

In [ ]:
import importlib, time, json, io, contextlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import shap

import ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, generate_bi_asset_data as gba
for m in (ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, gba):
    importlib.reload(m)
from ml_branch import MLBranch
from attention import build_dual_attention_bilstm
from feature_selection import BIAwareFeatureSelector
from feature_selection_aficv import AFICvFeatureSelector
from bi_fusion import CATEGORICAL_BI_VARS, CONTINUOUS_BI_VARS, BI_FEATURE_NAMES

# ---------------- Experiment configuration (fixed before running) ----------------
RHO = 0.4                      # asset-history link strength, chosen from W11 realism checks
FLEET = 2026                   # reference simulated fleet (= repo files)
SEEDS = [42, 43, 44]           # repeated CV (advisor point 4)
W = 30
RUL_MAX = 125
N_FOLDS = 5
N_CUTS = 3
INNER_VAL_FRAC = 0.2
CUT_SEED = 42                  # same truncation points as W11
ASSET_ROUTING = 'ml_only'      # static asset history goes to the ML branch only (see Section 4.1)

RF_PARAMS = dict(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2)   # benchmark, not tuned
BATCH, EPOCHS = 128, 100
DENSE_EPOCHS = 300           # dense fusion network (small, early-stopped; independent of QUICK)
CACHE_VERSION = 'v3b'          # v3 = rate-aware ML representation + operating age; b = BiLSTM collapse guard
EVAL_MIN_HISTORY = W          # score only truncation points with ≥ W observed cycles (see Stage 1)
MAX_DL_RETRIES = 2             # BiLSTM retrained with a new initialisation if it collapses to a constant (see train_dl)
REP = 'rate'                   # ML branch: 'rate' = sensors -> window statistics, slow BI and age -> last value, static -> once
                               #            'stats' = window statistics for every time-varying column (v2 design)
RUN_ROUTING_CHECK = False      # Section 4.1 (asset history to both branches); shown in v2, set True to repeat it
CORR_SEL = dict(variance_threshold=0.01, correlation_threshold=0.95)
AFICV_SEL = dict(base_learner='xgboost', n_folds=5, cumulative_threshold=0.90)

# QUICK = True: smoke test (tiny models, 1 seed). Use False for reported results.
QUICK = False
if QUICK:
    SEEDS = [42]; EPOCHS = 2
    RF_PARAMS = dict(n_estimators=30, max_depth=8, min_samples_split=5, min_samples_leaf=2)

GPU = tf.config.list_physical_devices('GPU')
XGB_EXTRA = {'device': 'cuda', 'tree_method': 'hist'} if GPU else {}
CACHE_DIR = CACHE_ROOT / ('quick' if QUICK else 'full')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"QUICK={QUICK} | seeds={SEEDS} | epochs={EPOCHS} | GPU={bool(GPU)} | cache={CACHE_DIR}")

In [ ]:
cols = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
def _load_train(ds):
    t = pd.read_csv(CMAPSS_DIR / f'train_{ds}.txt', sep=r'\s+', header=None, names=cols, engine='python')
    t['unit'] += gba.UNIT_OFFSET[ds]
    t['subset'] = ds
    return t
train = pd.concat([_load_train(ds) for ds in DATASETS], ignore_index=True)
train['total_life'] = train.groupby('unit')['cycle'].transform('max')
train['rul'] = (train['total_life'] - train['cycle']).clip(upper=RUL_MAX)
UNITS = np.sort(train['unit'].unique())
LIFE = train.groupby('unit')['total_life'].first()

SENSOR_COLS = [c for c in train.columns if c.startswith('sensor_')]
SETTING_COLS = [c for c in train.columns if c.startswith('setting_')]

# Operational BI (13 variables)
bi_op = pd.read_csv(BI_DIR / f'BI_{FLEET_TAG}_train.csv').rename(columns={'unit_id': 'unit'})
OP_VARS = [c for c in BI_FEATURE_NAMES if c in bi_op.columns]
OP_CAT = [c for c in CATEGORICAL_BI_VARS if c in OP_VARS]
OP_CONT = [c for c in CONTINUOUS_BI_VARS if c in OP_VARS]

# Asset-history BI (5 variables), reference fleet, rho fixed above
asset = pd.read_csv(ASSET_DIR / f"BI_{FLEET_TAG}_asset_{gba.rho_tag(RHO)}.csv").rename(columns={'unit_id': 'unit'})
ASSET_VARS = gba.ASSET_VARS
ASSET_CONT = ['prior_unplanned_removals', 'hours_since_overhaul_entry']
BATCH_LEVELS = [f"B{i}" for i in range(1, gba.N_BATCHES + 1)]

def build_table(op_df, asset_df):
    # One table with sensors + all BI, categorical variables one-hot encoded
    df = train[['unit', 'cycle', 'rul'] + SETTING_COLS + SENSOR_COLS].copy()
    df['cycles_in_service'] = df['cycle'].astype(float)      # operating age: CMMS usage counter, known online
    df = df.merge(
        op_df[['unit', 'cycle'] + OP_VARS], on=['unit', 'cycle'], how='left')
    for c in OP_CAT:
        df[c] = pd.Categorical(df[c].astype(int), categories=sorted(bi_op[c].unique()))
    df = pd.get_dummies(df, columns=OP_CAT, dtype=float)
    a = asset_df[['unit'] + ASSET_VARS].copy()
    a['component_batch'] = pd.Categorical(a['component_batch'], categories=BATCH_LEVELS)
    df = df.merge(pd.get_dummies(a, columns=['component_batch'], dtype=float), on='unit', how='left')
    return df.sort_values(['unit', 'cycle']).reset_index(drop=True)

# Falsification controls (same methods as W10/W11):
#  - operational BI: each variable permuted independently across all rows
#  - asset history: full records permuted across engines (stays constant within an engine)
op_shuf = bi_op.copy()
r_op = np.random.RandomState(42)
for c in OP_VARS:
    op_shuf[c] = r_op.permutation(op_shuf[c].values)
asset_shuf = asset.sort_values('unit').reset_index(drop=True).copy()
perm = np.random.default_rng(FLEET + 1000).permutation(len(asset_shuf))
asset_shuf[ASSET_VARS] = asset_shuf[ASSET_VARS].values[perm]

DATA = {'real': build_table(bi_op, asset),
        'asset_shuffled': build_table(bi_op, asset_shuf),
        'all_shuffled': build_table(op_shuf, asset_shuf)}
DF = DATA['real']
assert all(list(d.columns) == list(DF.columns) for d in DATA.values())

OP_COLS = [c for c in DF.columns if c in OP_CONT or c in OP_VARS or c.startswith(tuple(f"{v}_" for v in OP_CAT))]
ASSET_COLS = [c for c in DF.columns if c.startswith(tuple(ASSET_VARS))]
AGE_COLS = ['cycles_in_service']
# Feature sets chosen after W11 (Section 10): operating age and rate-aware BI. T = time in service (operating age).
FEATURE_SETS = {
    'S':          {'bi': [],                               'data': 'real'},          # sensors + settings only
    'S+T':        {'bi': AGE_COLS,                         'data': 'real'},          # + operating age (CMMS)
    'S+Op+A':     {'bi': OP_COLS + ASSET_COLS,             'data': 'real'},          # + 18 BI (13 operational + 5 asset history)
    'S+T+Op+A':   {'bi': AGE_COLS + OP_COLS + ASSET_COLS,  'data': 'real'},          # all enterprise data
    'S+T+Op~+A~': {'bi': AGE_COLS + OP_COLS + ASSET_COLS,  'data': 'all_shuffled'},  # control: age real, 18 BI shuffled
}
MAIN_FSETS = ['S', 'S+T', 'S+Op+A', 'S+T+Op+A']
FULL = 'S+T+Op+A'
CONT_COLS = SENSOR_COLS + SETTING_COLS + OP_CONT + ASSET_CONT + AGE_COLS
RATE_LAST = OP_COLS + AGE_COLS          # columns given to the ML branch by their last known value (REP = 'rate')
GROUP_OF = {**{c: 'sensor' for c in SENSOR_COLS + SETTING_COLS}, **{c: 'operating age' for c in AGE_COLS},
            **{c: 'operational BI' for c in OP_COLS}, **{c: 'asset BI' for c in ASSET_COLS}}

print(f"Fleet {FLEET_TAG}: " + ", ".join(f"{s} {n} engines" for s, n in train.groupby("subset").unit.nunique().items()))
print(f"Engines: {len(UNITS)}, rows: {len(DF)}, life [{LIFE.min()}, {LIFE.max()}]")
print(f"Columns: {len(SENSOR_COLS)} sensors + {len(SETTING_COLS)} settings + {len(OP_COLS)} operational BI "
      f"(13 variables) + {len(ASSET_COLS)} asset BI (5 variables) + operating age")

In [ ]:
BEST_PARAMS_PATH = (Path(f'/content/drive/MyDrive/PhD_results/W13_best_params{TAG_SUFFIX}.json') if IN_COLAB
                    else PROJECT_ROOT / 'results' / f'W13{TAG_SUFFIX}' / f'W13_best_params{TAG_SUFFIX}.json')
DL_FIXED = {'feature_attention_dim': 32, 'dropout_rate': 0.3}
if QUICK:
    _ml = {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.1, 'flatten': 'statistics'}
    _dl = {'lstm_units': 32, 'temporal_attention_dim': 64, 'dense_units': 32, 'learning_rate': 0.001, **DL_FIXED}
    TUNED = {f: {'ml': _ml, 'dl': _dl} for f in MAIN_FSETS}
else:
    if not BEST_PARAMS_PATH.exists():
        raise FileNotFoundError(f"{BEST_PARAMS_PATH} not found: run W13 (hyperparameter optimisation) first.")
    _b = json.loads(BEST_PARAMS_PATH.read_text())
    if _b.get('quick'):
        raise ValueError("W13 was run with QUICK = True: re-run W13 with QUICK = False.")
    TUNED = _b['params']

def params_for(fset):
    # Shuffled controls use the settings of the 18-BI configuration they are compared with
    return TUNED[fset if fset in TUNED else FULL]

display(pd.DataFrame({f: {**{f"ML {k}": v for k, v in TUNED[f]['ml'].items()},
                          **{f"DL {k}": v for k, v in TUNED[f]['dl'].items()}} for f in MAIN_FSETS}))

In [ ]:
folds = list(GroupKFold(n_splits=N_FOLDS).split(UNITS, groups=UNITS))

rng = np.random.default_rng(CUT_SEED)
cut_rows = []
for u in UNITS:
    life = int(LIFE[u])
    lo, hi = max(int(0.10 * life), 1), max(int(0.95 * life), 2)
    for k, c in enumerate(np.sort(rng.integers(lo, hi, N_CUTS))):
        cut_rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(cut_rows)

def inner_split(train_units, seed):
    r = np.random.default_rng(seed)
    val = r.choice(train_units, size=int(round(INNER_VAL_FRAC * len(train_units))), replace=False)
    return np.setdiff1d(train_units, val), np.sort(val)

assert all(len(set(UNITS[tr]) & set(UNITS[te])) == 0 for tr, te in folds)
print(f"{N_FOLDS} folds × {[len(te) for _, te in folds]} test engines; "
      f"{len(CUTS)} truncation points, true RUL {CUTS.true_rul.min()}–{CUTS.true_rul.max()}")

In [ ]:
# W14 truncation points: at least W_MAX = 60 observed cycles, so that every window length of study C is
# scored on real (not zero-padded) histories and all studies share one evaluation set.
W_MAX = 60
rng = np.random.default_rng(CUT_SEED)
cut_rows = []
for u in UNITS:
    life = int(LIFE[u])
    lo, hi = max(int(0.10 * life), W_MAX), max(int(0.95 * life), W_MAX + 1)
    for k, c in enumerate(np.sort(rng.integers(lo, hi, N_CUTS))):
        cut_rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(cut_rows)
print(f"{len(CUTS)} truncation points, observed history {CUTS.cut_cycle.min()}–{CUTS.cut_cycle.max()} cycles")

In [ ]:
STAT_SUFFIX = ('_mean', '_std', '_min', '_max', '_last', '_trend')
def is_static(c): return c.startswith(tuple(ASSET_VARS))
def original_name(flat):
    for s in STAT_SUFFIX:
        if flat.endswith(s):
            return flat[: -len(s)]
    return flat

def quiet(fn, *args, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kw)

def scale_fold(train_units, fset):
    d = DATA[FEATURE_SETS[fset]['data']].copy()
    sc = MinMaxScaler().fit(d.loc[d.unit.isin(train_units), CONT_COLS])
    d[CONT_COLS] = sc.transform(d[CONT_COLS])
    return d

def select_features(d, train_units, fset, method, exclude_asset=False):
    # Returns the selected column list, fitted on training engines only
    tr = d[d.unit.isin(train_units)]
    bi = FEATURE_SETS[fset]['bi']
    if exclude_asset:
        bi = [c for c in bi if c not in ASSET_COLS]
    if method == 'correlation':
        sel = BIAwareFeatureSelector(**CORR_SEL)
        return quiet(sel.select_features, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                     setting_cols=SETTING_COLS, exclude_cols=['unit', 'cycle', 'rul'])
    if method == 'aficv':
        sel = AFICvFeatureSelector(**AFICV_SEL)
        if bi:
            return quiet(sel.select_features_stratified, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                         setting_cols=SETTING_COLS, target_col='rul', group_col='unit')
        return quiet(sel.select_features, data=tr, feature_cols=SENSOR_COLS + SETTING_COLS,
                     target_col='rul', group_col='unit')
    raise ValueError(method)

def windows_train(d, units, feats):
    # PAD=False as in the article: windows ending at cycles W..T of each engine
    X, y, owner = [], [], []
    for u in units:
        g = d[d.unit == u]
        v = g[feats].to_numpy(np.float32)
        if len(v) < W:
            continue
        X.append(sliding_window_view(v, (W, v.shape[1]))[:, 0])
        y.append(g['rul'].to_numpy(np.float32)[W - 1:])
        owner += [u] * (len(v) - W + 1)
    return np.concatenate(X), np.concatenate(y), np.asarray(owner)

def windows_cuts(d, units, feats):
    # Last window before each truncation point (zero-padded if the engine is shorter than W)
    X, keys = [], []
    for u in units:
        v_full = d[d.unit == u][feats].to_numpy(np.float32)
        for _, c in CUTS[CUTS.unit == u].iterrows():
            v = v_full[:c.cut_cycle]
            if len(v) < W:
                v = np.vstack([np.zeros((W - len(v), len(feats)), np.float32), v])
            X.append(v[-W:]); keys.append((u, int(c.cut_id)))
    return np.asarray(X), keys

def static_rows(d, owners, static_cols):
    return d.groupby('unit')[static_cols].first().loc[list(owners)].to_numpy(np.float32)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def ml_design(ml, X, dyn, d, owners, stat):
    # ML-branch feature matrix. REP = 'rate': per-cycle columns (sensors, settings) -> window representation of
    # MLBranch; slow BI (refreshed every 10-50 cycles) and operating age -> last value in the window; static asset
    # history -> once per sample. REP = 'stats': window representation for every time-varying column.
    last = [i for i, c in enumerate(dyn) if REP == 'rate' and c in RATE_LAST]
    win = [i for i in range(len(dyn)) if i not in last]
    F = ml._flatten(X[:, :, win], [dyn[i] for i in win]); names = list(ml.feature_names_flat)
    if last:
        F = np.hstack([F, X[:, -1, last]]); names += [f"{dyn[i]}_last" for i in last]
    if stat:
        F = np.hstack([F, static_rows(d, owners, stat)]); names += stat
    return F, names

In [ ]:
def train_ml(d, fit_units, val_units, test_units, feats, seed, p):
    # XGBoost (tuned) and Random Forest benchmark on the same window representation
    dyn = [c for c in feats if not is_static(c)]
    stat = [c for c in feats if is_static(c)]
    Xtr, ytr, otr = windows_train(d, fit_units, dyn)
    Xva, yva, ova = windows_train(d, val_units, dyn)
    Xte, keys = windows_cuts(d, test_units, dyn)
    ml = MLBranch(model_type='xgboost', flatten_strategy=p['flatten'], random_state=seed,
                  n_estimators=p['n_estimators'], max_depth=p['max_depth'], learning_rate=p['learning_rate'], **XGB_EXTRA)
    Ftr, names = ml_design(ml, Xtr, dyn, d, otr, stat)
    Fva, _ = ml_design(ml, Xva, dyn, d, ova, stat)
    Fte, _ = ml_design(ml, Xte, dyn, d, [u for u, _ in keys], stat)
    t0 = time.perf_counter(); ml.model.fit(Ftr, ytr); t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); p_te = ml.model.predict(Fte); t_inf = (time.perf_counter() - t0) / len(Fte)
    p_va = ml.model.predict(Fva)
    y_te = CUTS.set_index(['unit', 'cut_id']).loc[keys, 'true_rul'].to_numpy(float)

    rf = RandomForestRegressor(random_state=seed, n_jobs=-1, **RF_PARAMS)
    t0 = time.perf_counter(); rf.fit(Ftr, ytr); t_train_rf = time.perf_counter() - t0
    p_rf = rf.predict(Fte)

    # Explainability on held-out truncation points: SHAP and permutation importance, per original variable
    t0 = time.perf_counter()
    sv = np.abs(shap.TreeExplainer(ml.model).shap_values(Fte)).mean(0)
    t_shap = time.perf_counter() - t0
    orig = np.array([original_name(n) for n in names])
    shap_imp = pd.Series(sv, index=orig).groupby(level=0).sum()
    base = rmse(y_te, p_te); r = np.random.default_rng(seed); perm = {}
    for var in np.unique(orig):
        idx = np.where(orig == var)[0]; inc = []
        for _ in range(10):
            Fp = Fte.copy(); Fp[:, idx] = Fp[r.permutation(len(Fp))][:, idx]
            inc.append(rmse(y_te, ml.model.predict(Fp)) - base)
        perm[var] = np.mean(inc)
    imp = pd.DataFrame({'shap_ml': shap_imp, 'perm_ml': pd.Series(perm)})
    return dict(pred=p_te, pred_rf=p_rf, keys=keys, val_rmse=rmse(yva, p_va), p_va=p_va, y_va=yva,
                t_train=t_train, t_train_rf=t_train_rf, t_inf=t_inf, t_shap=t_shap, imp=imp, n_feat=len(feats))

def train_dl(d, fit_units, val_units, test_units, feats, seed, p):
    # Collapse guard (W13: 17/192 BiLSTM fits stopped after 11-18 epochs predicting a constant, RMSE ≈ 38-40,
    # only in configurations with age or BI channels). A fit is declared collapsed when its inner-validation RMSE
    # exceeds 0.9 × the RMSE of a constant predictor (mean training RUL); it is then retrained with a new
    # initialisation, at most MAX_DL_RETRIES times. The rule uses inner-validation data only and is applied
    # identically to every configuration; the number of retries is stored in the run metadata.
    Xtr, ytr, _ = windows_train(d, fit_units, feats)
    Xva, yva, _ = windows_train(d, val_units, feats)
    Xte, keys = windows_cuts(d, test_units, feats)
    const_rmse = rmse(yva, np.full_like(yva, ytr.mean()))
    t0 = time.perf_counter()
    for retry in range(MAX_DL_RETRIES + 1):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed + 1000 * retry)
        model, att = build_dual_attention_bilstm(window_size=W, n_features=len(feats), **p)
        cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
              tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
        hist = model.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH, validation_data=(Xva, yva), callbacks=cb, verbose=0)
        if rmse(yva, model.predict(Xva, batch_size=1024, verbose=0).ravel()) <= 0.9 * const_rmse:
            break
    t_train = time.perf_counter() - t0
    t0 = time.perf_counter(); p_te = model.predict(Xte, batch_size=256, verbose=0).ravel()
    t_inf = (time.perf_counter() - t0) / len(Xte)
    p_va = model.predict(Xva, batch_size=1024, verbose=0).ravel()
    fw = att.predict(Xte, batch_size=256, verbose=0)['feature_weights']      # (n, W, n_feat)
    imp = pd.DataFrame({'attn_dl': pd.Series(np.asarray(fw).mean(axis=(0, 1)), index=feats)})
    return dict(pred=p_te, keys=keys, val_rmse=rmse(yva, p_va), p_va=p_va, y_va=yva, t_train=t_train, t_inf=t_inf,
                imp=imp, n_feat=len(feats), epochs=len(hist.history['loss']), retries=retry,
                collapsed=bool(rmse(yva, p_va) > 0.9 * const_rmse))

def dense_fusion(ml, dl, seed):
    # Learned fusion: small network on [y_DL, y_ML, |y_DL - y_ML|], trained on inner-validation predictions only
    assert np.allclose(ml['y_va'], dl['y_va'])
    # Inputs and target scaled by RUL_MAX for stable training; output rescaled to cycles
    feats = lambda a, b: np.column_stack([a, b, np.abs(a - b)]) / RUL_MAX
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    inp = tf.keras.layers.Input(shape=(3,))
    x = tf.keras.layers.Dense(8, activation='relu')(inp)
    x = tf.keras.layers.Dropout(0.2)(x)
    net = tf.keras.Model(inp, tf.keras.layers.Dense(1)(x))
    net.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='mse')
    net.fit(feats(dl['p_va'], ml['p_va']), ml['y_va'] / RUL_MAX, epochs=DENSE_EPOCHS, batch_size=64,
            validation_split=0.2, verbose=0,
            callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)])
    return net.predict(feats(dl['pred'], ml['pred']), verbose=0).ravel() * RUL_MAX

## 1. Ablation pipeline

In [ ]:
from attention import build_bilstm_ablation
import attention as _att; importlib.reload(_att); from attention import build_bilstm_ablation

ABL_SEED = 42
REF = dict(W=30, RUL_MAX=125, sel='design', att='dual')       # W12 design
ATT = {'dual': (True, True), 'feature': (True, False), 'temporal': (False, True), 'none': (False, False)}
ABL_DIR = CACHE_DIR / 'ablation_cut60'; ABL_DIR.mkdir(parents=True, exist_ok=True)
_BASE_RUL_MAX = RUL_MAX

def set_rul_cap(cap):
    # Re-clip the training target and the truth at the truncation points; truncation cycles do not change
    global RUL_MAX
    RUL_MAX = cap
    for dd in DATA.values():
        dd['rul'] = (dd['unit'].map(LIFE) - dd['cycle']).clip(upper=cap).astype(float)
    CUTS['true_rul'] = np.minimum(CUTS['unit'].map(LIFE) - CUTS['cut_cycle'], cap)

def select_variant(d, tr_u, fset, method, branch):
    # Feature selection for one branch; asset history never goes to the DL branch (ml_only routing)
    excl = branch == 'dl'
    if method == 'design':
        return select_features(d, tr_u, fset, 'correlation' if branch == 'ml' else 'aficv', exclude_asset=excl)
    if method in ('correlation', 'aficv_strat'):
        return select_features(d, tr_u, fset, 'correlation' if method == 'correlation' else 'aficv', exclude_asset=excl)
    bi = [c for c in FEATURE_SETS[fset]['bi'] if not (excl and c in ASSET_COLS)]
    allf = SENSOR_COLS + SETTING_COLS + bi
    if method == 'none':
        return allf
    if method == 'aficv_global':
        tr = d[d.unit.isin(tr_u)]
        return quiet(AFICvFeatureSelector(**AFICV_SEL).select_features, data=tr, feature_cols=allf,
                     target_col='rul', group_col='unit')
    raise ValueError(method)

def ml_light(d, fit_u, val_u, te_u, feats, seed, p):
    dyn = [c for c in feats if not is_static(c)]; stat = [c for c in feats if is_static(c)]
    Xtr, ytr, otr = windows_train(d, fit_u, dyn); Xva, yva, ova = windows_train(d, val_u, dyn)
    Xte, keys = windows_cuts(d, te_u, dyn)
    ml = MLBranch(model_type='xgboost', flatten_strategy=p['flatten'], random_state=seed,
                  n_estimators=p['n_estimators'], max_depth=p['max_depth'], learning_rate=p['learning_rate'], **XGB_EXTRA)
    Ftr, _ = ml_design(ml, Xtr, dyn, d, otr, stat); Fva, _ = ml_design(ml, Xva, dyn, d, ova, stat)
    Fte, _ = ml_design(ml, Xte, dyn, d, [u for u, _ in keys], stat)
    ml.model.fit(Ftr, ytr)
    return dict(pred=ml.model.predict(Fte), keys=keys, val_rmse=rmse(yva, ml.model.predict(Fva)))

def dl_light(d, fit_u, val_u, te_u, feats, seed, p, att):
    # Same collapse guard as W12 (train_dl)
    Xtr, ytr, _ = windows_train(d, fit_u, feats); Xva, yva, _ = windows_train(d, val_u, feats)
    Xte, keys = windows_cuts(d, te_u, feats)
    const = rmse(yva, np.full_like(yva, ytr.mean())); fa, ta = ATT[att]
    for retry in range(MAX_DL_RETRIES + 1):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed + 1000 * retry)
        model = build_bilstm_ablation(window_size=W, n_features=len(feats), feature_attention=fa, temporal_attention=ta, **p)
        cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
              tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
        h = model.fit(Xtr, ytr, epochs=EPOCHS, batch_size=BATCH, validation_data=(Xva, yva), callbacks=cb, verbose=0)
        v = rmse(yva, model.predict(Xva, batch_size=1024, verbose=0).ravel())
        if v <= 0.9 * const:
            break
    return dict(pred=model.predict(Xte, batch_size=256, verbose=0).ravel(), keys=keys, val_rmse=v,
                retries=retry, collapsed=bool(v > 0.9 * const), epochs=len(h.history['loss']))

def run_variant(study, fset, spec, k, seed=ABL_SEED):
    global W
    tag = f"{CACHE_VERSION}_{fset}_W{spec['W']}_R{spec['RUL_MAX']}_{spec['sel']}_{spec['att']}_s{seed}_f{k}"
    path = ABL_DIR / (tag.replace('+', 'p').replace('~', 'sh') + '.csv')
    if path.exists():
        out = pd.read_csv(path)
    else:
        W_old = W
        W = spec['W']; set_rul_cap(spec['RUL_MAX'])
        try:
            tr_idx, te_idx = folds[k]; tr_u, te_u = UNITS[tr_idx], UNITS[te_idx]
            fit_u, val_u = inner_split(tr_u, seed * 100 + k)
            d = scale_fold(tr_u, fset)
            f_ml = select_variant(d, tr_u, fset, spec['sel'], 'ml')
            f_dl = select_variant(d, tr_u, fset, spec['sel'], 'dl')
            hp = params_for(fset)
            ml = ml_light(d, fit_u, val_u, te_u, f_ml, seed, hp['ml'])
            dl = dl_light(d, fit_u, val_u, te_u, f_dl, seed, hp['dl'], spec['att'])
            alpha = (1 / dl['val_rmse']) / (1 / ml['val_rmse'] + 1 / dl['val_rmse'])
            out = pd.DataFrame(ml['keys'], columns=['unit', 'cut_id'])
            out['ml'], out['dl'] = ml['pred'], dl['pred']
            out['hybrid'] = alpha * out.dl + (1 - alpha) * out.ml
            out = out.merge(CUTS, on=['unit', 'cut_id'])
            out['alpha'], out['n_feat_ml'], out['n_feat_dl'] = alpha, len(f_ml), len(f_dl)
            out['dl_retries'], out['dl_collapsed'], out['dl_epochs'] = dl['retries'], dl['collapsed'], dl['epochs']
            out.to_csv(path, index=False)
        finally:
            W = W_old; set_rul_cap(_BASE_RUL_MAX)
    out[['study', 'fset', 'W', 'RUL_MAX', 'sel', 'att', 'seed', 'fold']] = (study, fset, spec['W'], spec['RUL_MAX'],
                                                                            spec['sel'], spec['att'], seed, k)
    return out
print("Ablation pipeline ready ✓")

## 2. Studies

In [ ]:
STUDIES = {
    'A. attention': [(f, {**REF, 'att': a}) for f in ['S', 'S+T'] for a in ['dual', 'feature', 'temporal', 'none']],
    'B. selection': [(FULL, {**REF, 'sel': s}) for s in ['design', 'none', 'correlation', 'aficv_global', 'aficv_strat']],
    'C. window':    [(f, {**REF, 'W': w}) for f in ['S', 'S+T'] for w in [15, 30, 45, 60]],
    'D. RUL cap':   [(f, {**REF, 'RUL_MAX': r}) for f in ['S', 'S+T'] for r in [100, 125, 150]],
}
n_new = len({(f, tuple(sorted(s.items()))) for v in STUDIES.values() for f, s in v})
print(f"{n_new} distinct (feature set, setting) pairs × {N_FOLDS} folds = {n_new * N_FOLDS} fold runs "
      f"(the reference runs are shared between studies)")

t0 = time.time(); R = []
for study, items in STUDIES.items():
    for fset, spec in items:
        parts = [run_variant(study, fset, spec, k) for k in range(N_FOLDS)]
        x = pd.concat(parts); R.append(x)
        print(f"{study:<13} {fset:<9} W={spec['W']:<3} cap={spec['RUL_MAX']:<4} sel={spec['sel']:<12} att={spec['att']:<9} | "
              f"XGB {rmse(x.true_rul, x.ml):5.2f}  DL {rmse(x.true_rul, x.dl):5.2f}  Hybrid {rmse(x.true_rul, x.hybrid):5.2f} "
              f"| retries {int((x.drop_duplicates('fold').dl_retries > 0).sum())} | {(time.time() - t0) / 60:.1f} min")
ABL = pd.concat(R, ignore_index=True)
ABL.to_csv(RESULTS / 'w14_ablation_predictions.csv', index=False)

## 3. Results

RMSE per branch and ΔRMSE of the hybrid against the reference of the same feature set (negative = the variant is better than the reference).

In [ ]:
def setting(spec_row):
    return {'A. attention': 'att', 'B. selection': 'sel', 'C. window': 'W', 'D. RUL cap': 'RUL_MAX'}[spec_row]
REF_VAL = {'att': 'dual', 'sel': 'design', 'W': 30, 'RUL_MAX': 125}

def eng(x, model='hybrid'):
    e = x[model] - x.true_rul
    return x.assign(e=e).groupby('unit').agg(abs_err=('e', lambda v: np.mean(np.abs(v))), sq_err=('e', lambda v: np.mean(v ** 2)))

def compare_x(a, b, model='hybrid'):
    ea, eb = eng(a, model), eng(b, model)
    ea, eb = ea.loc[eb.index], eb
    boot = np.random.default_rng(0).integers(0, len(eb), (2000, len(eb)))
    d = np.sqrt(ea.sq_err.mean()) - np.sqrt(eb.sq_err.mean())
    bt = np.sqrt(ea.sq_err.values[boot].mean(1)) - np.sqrt(eb.sq_err.values[boot].mean(1))
    diff = ea.abs_err.values - eb.abs_err.values
    p = stats.wilcoxon(diff).pvalue if np.any(diff != 0) else 1.0      # two-sided
    return d, np.percentile(bt, 2.5), np.percentile(bt, 97.5), p

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for i, j in enumerate(o):
        run = max(run, (len(p) - i) * p[j]); adj[j] = min(run, 1)
    return adj

rows = []
for study, items in STUDIES.items():
    key = setting(study); fam = []
    for fset, spec in items:
        x = ABL[(ABL.study == study) & (ABL.fset == fset) & (ABL[key] == spec[key])]
        r = {'study': study, 'Features': fset, 'setting': f"{key} = {spec[key]}",
             'RMSE XGB': rmse(x.true_rul, x.ml), 'RMSE DL': rmse(x.true_rul, x.dl), 'RMSE Hybrid': rmse(x.true_rul, x.hybrid),
             'DL retries': int((x.drop_duplicates('fold').dl_retries > 0).sum())}
        if spec[key] != REF_VAL[key] and study != 'D. RUL cap':     # caps change the metric itself: compared in 3.2
            ref = ABL[(ABL.study == study) & (ABL.fset == fset) & (ABL[key] == REF_VAL[key])]
            r['Δ Hybrid vs ref'], r['CI low'], r['CI high'], r['p'] = compare_x(x, ref)
            fam.append(len(rows))
        rows.append(r)
    if fam:
        adj = holm([rows[i]['p'] for i in fam])
        for i, a in zip(fam, adj):
            rows[i]['p_holm'] = a
ABL_TAB = pd.DataFrame(rows)
ABL_TAB['sig'] = pd.cut(ABL_TAB.p_holm, [-1, 0.001, 0.01, 0.05, 2], labels=['***', '**', '*', 'ns'])
ABL_TAB.to_csv(RESULTS / 'w14_ablation_summary.csv', index=False)
display(ABL_TAB.round(3))

### 3.1 Feature counts selected in study B

In [ ]:
nf = ABL[ABL.study == 'B. selection'].drop_duplicates(['sel', 'fold']).groupby('sel')[['n_feat_ml', 'n_feat_dl']].agg(['min', 'max'])
display(nf)
nf.to_csv(RESULTS / 'w14_selection_feature_counts.csv')

### 3.2 Study D — gain of operating age at each RUL cap

If most of the gain of operating age came from the cap (early life easy to predict from age when the RUL is capped), the gain would shrink markedly when the cap is raised.

In [ ]:
rows = []
for cap in [100, 125, 150]:
    s = ABL[(ABL.study == 'D. RUL cap') & (ABL.fset == 'S') & (ABL.RUL_MAX == cap)]
    t = ABL[(ABL.study == 'D. RUL cap') & (ABL.fset == 'S+T') & (ABL.RUL_MAX == cap)]
    for model in ['ml', 'dl', 'hybrid']:
        d, lo, hi, p = compare_x(t, s, model)
        rows.append({'RUL cap': cap, 'model': model, 'RMSE S': rmse(s.true_rul, s[model]), 'RMSE S+T': rmse(t.true_rul, t[model]),
                     'Δ age (S+T − S)': d, 'CI low': lo, 'CI high': hi, 'relative gain %': 100 * d / rmse(s.true_rul, s[model]), 'p': p})
CAP = pd.DataFrame(rows)
CAP['p_holm'] = holm(CAP.p)
CAP.to_csv(RESULTS / 'w14_rul_cap_age_gain.csv', index=False)
display(CAP.round(3))

# Same question for the window length: is the age gain smaller with longer windows?
rows = []
for w in [15, 30, 45, 60]:
    s = ABL[(ABL.study == 'C. window') & (ABL.fset == 'S') & (ABL.W == w)]
    t = ABL[(ABL.study == 'C. window') & (ABL.fset == 'S+T') & (ABL.W == w)]
    d, lo, hi, p = compare_x(t, s)
    rows.append({'W': w, 'Hybrid S': rmse(s.true_rul, s.hybrid), 'Hybrid S+T': rmse(t.true_rul, t.hybrid),
                 'Δ age': d, 'CI low': lo, 'CI high': hi})
WIN = pd.DataFrame(rows).round(3)
WIN.to_csv(RESULTS / 'w14_window_age_gain.csv', index=False)
display(WIN)

## 4. Figure

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
for ax, study in zip(axes, STUDIES):
    key = setting(study)
    t = ABL_TAB[ABL_TAB.study == study]
    for fset, g in t.groupby('Features', sort=False):
        ax.plot([s.split(' = ')[1] for s in g.setting], g['RMSE Hybrid'], marker='o', label=fset)
    ax.set_title(study); ax.set_ylabel('Hybrid RMSE (cycles)'); ax.set_xlabel(key); ax.legend()
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w14_ablation.png', dpi=150); plt.show()

if SAVE_DIR is not None:
    for f in RESULTS.iterdir():
        if f.is_file() and f.name.startswith(('w14', 'fig_w14')):
            shutil.copy(f, SAVE_DIR / f.name)
    print(f"Results copied to {SAVE_DIR}")

## 5. Reading the results

- **A. Attention:** report the hybrid and the BiLSTM RMSE for each variant. If removing attention does not degrade the model, say so: attention then serves interpretability rather than accuracy.
- **B. Selection:** compare the W12 design with no selection; a gain supports the BI-aware selection step.
- **C. Window:** a flat curve around W = 30 supports the choice; report the age gain at each W (3.2).
- **D. RUL cap:** RMSEs at different caps are not comparable with each other (the target changes); compare the age gain within each cap. A gain that persists at cap 150 shows that operating age is not only exploiting the capped early-life region.
- **Limitations:** one seed; hyperparameters tuned for the reference settings only.